<a href="https://colab.research.google.com/github/abasovmanaf-3267/project_polars/blob/main/Polars_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring and Analyzing Movie Ratings with Polars: Project



| File        | Column     | Description                                                                 |
|------------|-----------|------------------------------------------------------------------------------|
| movies.csv | movieId   | ID number that uniquely identifies each movie                                |
| movies.csv | title     | Name of the movie (may include year in parentheses)                          |
| movies.csv | genres    | Pipe-separated list of genres (e.g., Comedy|Drama)                           |
| ratings.csv| userId    | ID number that uniquely identifies each user                                 |
| ratings.csv| movieId   | ID number that uniquely identifies each movie                                |
| ratings.csv| rating    | Rating score from 0.5 to 5.0 (half-star increments allowed)                  |
| ratings.csv| timestamp | Unix timestamp when the rating was given                                     |
| tags.csv   | userId    | ID number that uniquely identifies each user                                 |
| tags.csv   | movieId   | ID number that uniquely identifies each movie                                |
| tags.csv   | tag       | User-generated tag (string, e.g., 'funny')                                    |
| tags.csv   | timestamp | Unix timestamp when the tag was added                                        |


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import polars as pl

In [ ]:
movies = pl.read_csv('/content/drive/MyDrive/Polars_project/movies.csv')
ratings = pl.read_csv('/content/drive/MyDrive/Polars_project/ratings.csv')
tags = pl.read_csv('/content/drive/MyDrive/Polars_project/tags.csv')

# **Movies DATA FRAME**

In [ ]:
movies.head()

movieId,title,genres
i64,str,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…"
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance"""
5,"""Father of the Bride Part II (1…","""Comedy"""


# Movies Data setimiz 9742 setr ve 3 sütundan ibarətdir. Sütunlarımız "int" və "str" formatındadır və dəyişikliyə ehtiyac yoxdur

In [ ]:
movies.glimpse()

Rows: 9742
Columns: 3
$ movieId <i64> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
$ title   <str> 'Toy Story (1995)', 'Jumanji (1995)', 'Grumpier Old Men (1995)', 'Waiting to Exhale (1995)', 'Father of the Bride Part II (1995)', 'Heat (1995)', 'Sabrina (1995)', 'Tom and Huck (1995)', 'Sudden Death (1995)', 'GoldenEye (1995)'
$ genres  <str> 'Adventure|Animation|Children|Comedy|Fantasy', 'Adventure|Children|Fantasy', 'Comedy|Romance', 'Comedy|Drama|Romance', 'Comedy', 'Action|Crime|Thriller', 'Comedy|Romance', 'Adventure|Children', 'Action', 'Action|Adventure|Thriller'



#Heç bir dublikat, null və nan dəyər tapılmadı

In [ ]:
movies.is_duplicated().sum()

0

In [ ]:
movies.null_count().sum()

movieId,title,genres
u32,u32,u32
0,0,0


In [ ]:
movies["movieId"].is_nan().sum()

0

# Movie id sütun adı yeniləndi

In [ ]:
movies = movies.rename({"movieId": "movie_id"})

# **Ratings DATA FRAME**

In [ ]:
ratings.head()

userId,movieId,rating,timestamp
i64,i64,f64,i64
1,1,4.0,964982703
1,3,4.0,964981247
1,6,4.0,964982224
1,47,5.0,964983815
1,50,5.0,964982931


#Ratings Data setimiz 100836 setr ve 4 sütundan ibarətdir. Sütunlarımız "int" və "float" formatındadır və dəyişikliyə ehtiyac yoxdur



In [ ]:
ratings.glimpse()

Rows: 100836
Columns: 4
$ userId    <i64> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1
$ movieId   <i64> 1, 3, 6, 47, 50, 70, 101, 110, 151, 157
$ rating    <f64> 4.0, 4.0, 4.0, 5.0, 5.0, 3.0, 5.0, 4.0, 5.0, 5.0
$ timestamp <i64> 964982703, 964981247, 964982224, 964983815, 964982931, 964982400, 964980868, 964982176, 964984041, 964984100



#Heç bir dublikat, null və nan dəyər tapılmadı

In [ ]:
ratings.is_duplicated().sum()

0

In [ ]:
ratings.null_count().sum()

userId,movieId,rating,timestamp
u32,u32,u32,u32
0,0,0,0


In [ ]:
ratings.select(pl.col("*").is_nan()).sum()

userId,movieId,rating,timestamp
u32,u32,u32,u32
0,0,0,0


In [ ]:
ratings=ratings.rename({"userId":"user_id","movieId":"movie_id"})

# **Tags DATA FRAME**

In [ ]:
tags.head()

userId,movieId,tag,timestamp
i64,i64,str,i64
2,60756,"""funny""",1445714994
2,60756,"""Highly quotable""",1445714996
2,60756,"""will ferrell""",1445714992
2,89774,"""Boxing story""",1445715207
2,89774,"""MMA""",1445715200


#Tags Data setimiz 3683 setr ve 4 sütundan ibarətdir. Sütunlarımız "int" və "str" formatındadır və dəyişikliyə ehtiyac yoxdur



In [ ]:
tags.glimpse()

Rows: 3683
Columns: 4
$ userId    <i64> 2, 2, 2, 2, 2, 2, 2, 2, 2, 7
$ movieId   <i64> 60756, 60756, 60756, 89774, 89774, 89774, 106782, 106782, 106782, 48516
$ tag       <str> 'funny', 'Highly quotable', 'will ferrell', 'Boxing story', 'MMA', 'Tom Hardy', 'drugs', 'Leonardo DiCaprio', 'Martin Scorsese', 'way too long'
$ timestamp <i64> 1445714994, 1445714996, 1445714992, 1445715207, 1445715200, 1445715205, 1445715054, 1445715051, 1445715056, 1169687325



#Heç bir dublikat, null və nan dəyər tapılmadı



In [ ]:
tags.null_count().sum()

userId,movieId,tag,timestamp
u32,u32,u32,u32
0,0,0,0


In [ ]:
tags.is_duplicated().sum()

0

In [ ]:
tags.select(pl.col(["userId","movieId","timestamp"]).is_nan()).sum()

userId,movieId,timestamp
u32,u32,u32
0,0,0


# Movie id , user id ve timestamp sütun adı yeniləndi
Qeyd: Timestamp adının dəyişdirilməsində məqsəd rating cədvəlindədə eyni adda sütun olması və cədvəllərin birləşdirilməsi zamanı sütunların qarışıqlıq yaratmaması üçün.

In [ ]:
tags=tags.rename({"userId":"user_id","movieId":"movie_id","timestamp":"timestamp_tags"})

#Cədvəllərin id sütunları üzrə birləşdirilməsi

In [ ]:
merged_frame = ratings.join(movies, on="movie_id", how="inner")
fin3 = merged_frame.join(tags, on=["movie_id", "user_id"], how="inner")

In [ ]:
fin3.head()

user_id,movie_id,rating,timestamp,title,genres,tag,timestamp_tags
i64,i64,f64,i64,str,str,str,i64
2,60756,5.0,1445714980,"""Step Brothers (2008)""","""Comedy""","""funny""",1445714994
2,60756,5.0,1445714980,"""Step Brothers (2008)""","""Comedy""","""Highly quotable""",1445714996
2,60756,5.0,1445714980,"""Step Brothers (2008)""","""Comedy""","""will ferrell""",1445714992
2,89774,5.0,1445715189,"""Warrior (2011)""","""Drama""","""Boxing story""",1445715207
2,89774,5.0,1445715189,"""Warrior (2011)""","""Drama""","""MMA""",1445715200


# Dəyərlər yoxlanıldı.Minumum rating 0.5 maksiumum dəyər 5 olaraq göstərildi.

In [ ]:
fin3.select([pl.col("rating").min().alias("min_rating"),
         pl.col("rating").max().alias("max_rating")])


min_rating,max_rating
f64,f64
0.5,5.0


#Ən çox verilən rating sayı 999 olaraq çoxdan aza doğru sıralandı.

In [ ]:
fin3["rating"].value_counts(sort=True)

rating,count
f64,u32
4.0,999
5.0,883
3.5,577
4.5,496
3.0,274
2.0,102
2.5,80
1.0,31
1.5,26


#Burada janr dəyərləri split ve explode metodları ilə ayrılıb fərdiləşdirildi ortalama rating üzrə qruplaşdırıldı.

In [ ]:
fin3.with_columns(
    pl.col("genres").str.split("|")).explode("genres").group_by("genres").agg(pl.col("rating").mean())

genres,rating
str,f64
"""Comedy""",4.036199
"""Drama""",4.147
"""Crime""",4.385751
"""War""",3.872549
"""IMAX""",4.003289
…,…
"""Horror""",3.911483
"""Musical""",3.905983
"""Western""",4.516949


#Bu cədvəldə hər bir istifadəçi üzrə nə qədər min,max,ortalama rating xalı verilib onlar göstərilib

In [ ]:
fin3.group_by("user_id").agg([pl.col("rating").min().alias("min_rating"),
    pl.col("rating").max().alias("max_rating"),
    pl.col("rating").mean().alias("mean_rating")])

user_id,min_rating,max_rating,mean_rating
i64,f64,f64,f64
2,5.0,5.0,5.0
18,3.0,5.0,4.125
21,0.5,0.5,0.5
106,5.0,5.0,5.0
125,4.0,5.0,4.916667
…,…,…,…
567,0.5,5.0,3.917824
573,0.5,5.0,3.791667
599,5.0,5.0,5.0


#Ən çox rating xalı verən 5 istifadəçi

In [ ]:
fin3.group_by("user_id").agg(pl.col("rating").count().alias("rating_count")).sort("rating_count", descending=True).head(5)

user_id,rating_count
i64,u32
474,1414
567,432
62,370
599,323
477,267


#Bu analizdə əvvəlcə hər bir filmin neçə reytinq aldığını hesablayırıq. Daha sonra şərtə əsasən filmləri iki qrupa bölürük. Ən çox reytinq alan top 10 film və ən az reytinq alan bottom 10 film olaraq və müqaisə edirik

In [ ]:
top10=fin3.group_by("movie_id").agg(pl.col("rating").count()).sort(pl.col("rating"),descending=True).head(10)

bottom10=fin3.group_by("movie_id").agg(
    pl.col("rating").count().alias("rating_count")).filter(pl.col("rating_count") >= 5).sort(pl.col("rating_count")).head(10)

print(top10,bottom10)

shape: (10, 2)
┌──────────┬────────┐
│ movie_id ┆ rating │
│ ---      ┆ ---    │
│ i64      ┆ u32    │
╞══════════╪════════╡
│ 296      ┆ 181    │
│ 2959     ┆ 54     │
│ 924      ┆ 41     │
│ 293      ┆ 35     │
│ 1732     ┆ 32     │
│ 7361     ┆ 30     │
│ 4878     ┆ 28     │
│ 79132    ┆ 26     │
│ 135536   ┆ 19     │
│ 260      ┆ 18     │
└──────────┴────────┘ shape: (10, 2)
┌──────────┬──────────────┐
│ movie_id ┆ rating_count │
│ ---      ┆ ---          │
│ i64      ┆ u32          │
╞══════════╪══════════════╡
│ 1343     ┆ 5            │
│ 152711   ┆ 5            │
│ 104944   ┆ 5            │
│ 903      ┆ 5            │
│ 4223     ┆ 5            │
│ 6333     ┆ 5            │
│ 68157    ┆ 5            │
│ 2329     ┆ 5            │
│ 6818     ┆ 5            │
│ 63992    ┆ 5            │
└──────────┴──────────────┘


In [ ]:
fin3.group_by("movie_id").agg(
    pl.col("rating").count().alias("rating_count")
).filter(pl.col("rating_count") >= 5).sort(pl.col("rating_count")).head(10)


movie_id,rating_count
i64,u32
903,5
2329,5
2731,5
152711,5
56367,5
63992,5
6852,5
104944,5
1343,5


In [ ]:
rtng_count = fin3.group_by("movie_id",).agg(pl.len().alias("count"))
dist = rtng_count.group_by("count").agg(pl.len().alias("number_of_movies")
).sort("count")

print(dist)

shape: (26, 2)
┌───────┬──────────────────┐
│ count ┆ number_of_movies │
│ ---   ┆ ---              │
│ u32   ┆ u32              │
╞═══════╪══════════════════╡
│ 1     ┆ 933              │
│ 2     ┆ 223              │
│ 3     ┆ 94               │
│ 4     ┆ 45               │
│ 5     ┆ 38               │
│ …     ┆ …                │
│ 32    ┆ 1                │
│ 35    ┆ 1                │
│ 41    ┆ 1                │
│ 54    ┆ 1                │
│ 181   ┆ 1                │
└───────┴──────────────────┘


# Bu cədvəldə ən çox rating dəyərləndirilməsi edilən 20 film göstərilib

In [ ]:
fin3.group_by("movie_id","title").agg(pl.col("rating").count()).sort("rating",descending=True).head(20)

movie_id,title,rating
i64,str,u32
296,"""Pulp Fiction (1994)""",181
2959,"""Fight Club (1999)""",54
924,"""2001: A Space Odyssey (1968)""",41
293,"""Léon: The Professional (a.k.a.…",35
1732,"""Big Lebowski, The (1998)""",32
…,…,…
5673,"""Punch-Drunk Love (2002)""",13
4226,"""Memento (2000)""",13
71899,"""Mary and Max (2009)""",13


#Cədvəldən görünür ki, datasetdə Drama janrı ən çox sevilən film növüdür(2000 film) və digər janrları demək olar ki, iki dəfə üstələyir.

In [ ]:
df_genres = fin3.with_columns(pl.col("genres").str.split("|")).explode("genres")
print( df_genres["genres"].n_unique())
print(df_genres.group_by("genres").len().sort("len", descending=True).head(10))


20
shape: (10, 2)
┌───────────┬──────┐
│ genres    ┆ len  │
│ ---       ┆ ---  │
│ str       ┆ u32  │
╞═══════════╪══════╡
│ Drama     ┆ 2000 │
│ Comedy    ┆ 1105 │
│ Thriller  ┆ 1065 │
│ Action    ┆ 788  │
│ Crime     ┆ 779  │
│ Sci-Fi    ┆ 641  │
│ Romance   ┆ 585  │
│ Adventure ┆ 582  │
│ Mystery   ┆ 388  │
│ Fantasy   ┆ 283  │
└───────────┴──────┘


# Hər janr üzrə film sayı və orta ratinglərin müqayisəsi göstərir ki, ən çox filmə sahib olan əsas janrlar ümumilikdə sabit və orta səviyyədə ratinglərə malikdir.

In [ ]:
genres_split= fin3.with_columns(pl.col("genres").str.split("|")).explode("genres")
genres_split.group_by("genres").agg(pl.col("movie_id").count().alias("movie_count"),
pl.col("rating").mean().round().alias("rating_mean")).sort("movie_count","rating_mean",descending=True).head(20)

genres,movie_count,rating_mean
str,u32,f64
"""Drama""",2000,4.0
"""Comedy""",1105,4.0
"""Thriller""",1065,4.0
"""Action""",788,4.0
"""Crime""",779,4.0
…,…,…
"""Musical""",117,4.0
"""Documentary""",97,4.0
"""Western""",59,5.0


#Nəticələr göstərir ki, tag sayları arasında böyük fərq var: əksər filmlərin etiketləri az olsa da, bəzi seçilmiş filmlər çoxlu sayda istifadəçi tagi toplayıb.

In [ ]:
#Hər bir film üçün düşən orta tэг (etiket) sayını hesablayıb çap edin, həmçinin hər film üzrə minimum və maksimum tэг sayını göstərin.
tags=fin3.group_by("movie_id").agg(pl.col("tag").count().alias("tag_count"))
tags.select(pl.col("tag_count").mean().alias("tag_mean"),
            pl.col("tag_count").min() .alias("tag_min"),
            pl.col("tag_count").max() .alias("tag_max"))


tag_mean,tag_min,tag_max
f64,u32,u32
2.374317,1,181


#Ən çox istifadə olunan top 20 tagin təhlili göstərir ki, istifadəçilər filmləri əsasən ümumi janrlar xüsusi mövzular və ya vizual üslublar üzrə qruplaşdırmağa üstünlük verirlər

In [ ]:
fin3.group_by("tag").count().sort("count",descending=True).head(20)

/tmp/ipykernel_1439/2524098303.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  fin3.group_by("tag").count().sort("count",descending=True).head(20)


tag,count
str,u32
"""In Netflix queue""",55
"""atmospheric""",36
"""funny""",23
"""superhero""",23
"""surreal""",23
…,…
"""crime""",18
"""music""",16
"""mental illness""",16
